# Section 10 — Vanilla FT + SSIAT/SAPT + SAFE (Vast H100/A100)

Runs all 3 continual learning methods sequentially under the correct JEPA loss.
2000 clips/session, 3 sessions, 3 epochs. Combined t-SNE + neighbour grid at end.

In [ ]:
# ── Cell 0: Install ───────────────────────────────────────────────────────────
import os
if not os.path.exists('/workspace/vjepa2'):
    os.system('git clone https://github.com/facebookresearch/vjepa2.git /workspace/vjepa2')
os.chdir('/workspace/vjepa2')
os.system('pip install -e ".[dev]" -q')
os.system('pip install -q webdataset huggingface_hub opencv-python-headless scikit-learn')
print('Install complete.')

In [ ]:
# ── Cell 1: Auth ──────────────────────────────────────────────────────────────
from huggingface_hub import login
login("YOUR_HF_TOKEN_HERE")

In [ ]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────────
import os

CKPT_PATH  = "/workspace/vjepa2_1_vitb_dist_vitG_384.pt"
CKPT_DIR   = "/workspace/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

HF_DATASET        = "anonymousML123/walkindia-200k"
SESSION_SHARDS    = [(0, 37), (38, 75), (76, 99)]
VAL_SHARD         = 100
CLIPS_PER_SESSION = 2000
VAL_CLIPS         = 50

IMG_SIZE        = 224
FRAMES_PER_CLIP = 16
BATCH_SIZE      = 8
NUM_EPOCHS      = 3
LR_BACKBONE     = 5e-5
LR_PREDICTOR    = 1e-4
EMA_MOMENTUM    = 0.996
GRAD_CLIP       = 1.0

TARGET_MASK_RATIO = 0.20
NUM_MASK_BLOCKS   = 4

LORA_RANK     = 8
TARGET_BLOCKS = [8, 9, 10, 11]
EVAL_K        = 10

N_QUERIES    = 5
N_NEIGHBOURS = 5

print("Config ready.")

In [ ]:
# ── Cell 3: Download V-JEPA checkpoint ────────────────────────────────────────
import subprocess
if not os.path.exists(CKPT_PATH):
    print("Downloading V-JEPA 2.1 ViT-B checkpoint...")
    subprocess.run(["wget", "-q",
        "https://dl.fbaipublicfiles.com/vjepa2/vjepa2_1_vitb_dist_vitG_384.pt",
        "-O", CKPT_PATH], check=True)
    print("Done.")
else:
    print(f"Checkpoint found: {CKPT_PATH}")

In [ ]:
# ── Cell 4: Imports + encoder ─────────────────────────────────────────────────
import sys, torch, copy, time, numpy as np
import torch.nn as nn
import torch.nn.functional as F
import webdataset as wds
import tempfile, cv2
from huggingface_hub import hf_hub_download
from torch.utils.data import DataLoader, IterableDataset

sys.path.insert(0, '/workspace/vjepa2')
try:
    import app.vjepa_2_1.models.vision_transformer as vit_module
except ModuleNotFoundError:
    hub_path = "/root/.cache/torch/hub/facebookresearch_vjepa2_main"
    sys.path.insert(0, hub_path)
    import app.vjepa_2_1.models.vision_transformer as vit_module

import src.datasets.utils.video.transforms as video_transforms
import src.datasets.utils.video.volume_transforms as volume_transforms

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


def build_encoder():
    model = vit_module.vit_base(
        patch_size=16, img_size=(IMG_SIZE, IMG_SIZE),
        num_frames=64, tubelet_size=2, use_sdpa=True,
        use_SiLU=False, wide_SiLU=True, uniform_power=False,
        use_rope=True, img_temporal_dim_size=1, interpolate_rope=True,
    )
    ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=True)
    sd   = {k.replace("module.", "").replace("backbone.", ""): v
            for k, v in ckpt["ema_encoder"].items()}
    msg  = model.load_state_dict(sd, strict=True)
    print(f"Encoder loaded: {msg}")
    return model


class JEPAPredictor(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, embed_dim * 2),
            nn.GELU(),
            nn.Linear(embed_dim * 2, embed_dim),
        )
    def forward(self, x):
        return self.net(x.mean(dim=1))


print("Imports done.")

In [ ]:
# ── Cell 5: Transforms ────────────────────────────────────────────────────────
MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)

def build_transform(augment=False):
    short = int(256.0 / 224 * IMG_SIZE)
    ops   = [video_transforms.Resize(short, interpolation="bilinear")]
    ops  += [video_transforms.RandomCrop((IMG_SIZE, IMG_SIZE))
              if augment else video_transforms.CenterCrop((IMG_SIZE, IMG_SIZE))]
    ops  += [volume_transforms.ClipToTensor(),
              video_transforms.Normalize(mean=MEAN, std=STD)]
    return video_transforms.Compose(ops)

transform = build_transform(augment=False)
print("Transforms ready.")

In [ ]:
# ── Cell 6: Data loading ──────────────────────────────────────────────────────
def decode_mp4(mp4_bytes, n_frames=FRAMES_PER_CLIP):
    """Returns (tensor [C,T,H,W], mid_frame [H,W,3]) or (None, None)."""
    with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as f:
        f.write(mp4_bytes); tmp = f.name
    try:
        cap   = cv2.VideoCapture(tmp)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total < 2: return None, None
        indices = np.linspace(0, total - 1, n_frames, dtype=int)
        frames  = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ret, frame = cap.read()
            if not ret: return None, None
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        cap.release()
        mid   = frames[len(frames) // 2]
        video = np.stack(frames)
        return transform(torch.from_numpy(video).permute(0, 3, 1, 2)), mid
    except Exception:
        return None, None
    finally:
        os.unlink(tmp)


class ShardDataset(IterableDataset):
    def __init__(self, shard_range, max_clips=-1):
        self.shard_range = shard_range
        self.max_clips   = max_clips

    def __iter__(self):
        count = 0
        start, end = self.shard_range
        for shard_idx in range(start, end + 1):
            if self.max_clips > 0 and count >= self.max_clips:
                return
            print(f"  [data] downloading shard {shard_idx:05d}...", flush=True)
            local = hf_hub_download(
                HF_DATASET,
                filename=f"data/train-{shard_idx:05d}.tar",
                repo_type="dataset"
            )
            print(f"  [data] shard {shard_idx:05d} ready, decoding...", flush=True)
            for sample in wds.WebDataset(local, shardshuffle=False):
                if self.max_clips > 0 and count >= self.max_clips:
                    return
                mp4 = sample.get("mp4")
                if mp4 is None: continue
                tensor, _ = decode_mp4(mp4)
                if tensor is None: continue
                count += 1
                if count % 200 == 0:
                    print(f"  [data] {count} clips decoded", flush=True)
                yield tensor


def make_loader(shard_range, max_clips=-1):
    return DataLoader(
        ShardDataset(shard_range, max_clips=max_clips),
        batch_size=BATCH_SIZE, num_workers=0, pin_memory=True
    )


print("Data loading ready.")

In [ ]:
# ── Cell 7: JEPA loss ─────────────────────────────────────────────────────────
def sample_masks(n_tokens, device):
    target     = torch.zeros(n_tokens, dtype=torch.bool, device=device)
    block_size = max(1, int(TARGET_MASK_RATIO * n_tokens / NUM_MASK_BLOCKS))
    for _ in range(NUM_MASK_BLOCKS):
        start = torch.randint(0, max(1, n_tokens - block_size), (1,)).item()
        target[start: start + block_size] = True
    context = ~target
    if context.sum() == 0:
        context[0] = True; target[0] = False
    return context, target


def jepa_loss(student, teacher, predictor, videos, device):
    with torch.no_grad():
        all_tok = teacher(videos)
    N = all_tok.shape[1]
    ctx_mask, tgt_mask = sample_masks(N, device)
    teacher_tgt        = all_tok[:, tgt_mask, :].mean(dim=1).detach()
    student_tok        = student(videos)
    prediction         = predictor(student_tok[:, ctx_mask, :])
    return F.mse_loss(prediction, teacher_tgt)


print("JEPA loss ready.")

In [ ]:
# ── Cell 8: Evaluation — Cycle@K ─────────────────────────────────────────────
@torch.no_grad()
def extract_embeddings_and_frames(encoder, shard_idx=VAL_SHARD, max_clips=VAL_CLIPS):
    """Returns embeddings [N,D] and mid_frames list for visualization."""
    encoder.eval()
    embs, frames = [], []
    print(f"  [eval] downloading val shard {shard_idx:05d}...", flush=True)
    local = hf_hub_download(
        HF_DATASET, filename=f"data/train-{shard_idx:05d}.tar", repo_type="dataset")
    print(f"  [eval] extracting {max_clips} embeddings...", flush=True)
    count = 0
    for sample in wds.WebDataset(local, shardshuffle=False):
        if count >= max_clips: break
        mp4 = sample.get("mp4")
        if mp4 is None: continue
        tensor, mid = decode_mp4(mp4)
        if tensor is None: continue
        feats = encoder(tensor.unsqueeze(0).to(DEVICE)).mean(dim=1)
        feats = F.normalize(feats, dim=-1)
        embs.append(feats.cpu())
        frames.append(mid)
        count += 1
    embs = torch.cat(embs, dim=0)[:max_clips]
    return embs, frames


def cycle_at_k(embs, k=EVAL_K):
    N   = embs.shape[0]
    sim = embs @ embs.T
    sim.fill_diagonal_(-1e9)
    top1_b = sim.argmax(dim=1)
    topk_b = sim[top1_b].topk(k, dim=1).indices
    hits   = (topk_b == torch.arange(N).unsqueeze(1)).any(dim=1)
    return hits.float().mean().item()


def evaluate(encoder):
    t0             = time.time()
    embs, frames   = extract_embeddings_and_frames(encoder)
    cycle          = cycle_at_k(embs)
    elapsed        = time.time() - t0
    print(f"  [eval] Cycle@{EVAL_K}={cycle:.3f}  ({elapsed:.0f}s)", flush=True)
    return {"cycle_at_k": cycle}, embs, frames


print("Evaluation helpers ready.")

In [ ]:
# ── Cell 9: Shared utilities ──────────────────────────────────────────────────
class LoRALinear(nn.Module):
    def __init__(self, original, rank=LORA_RANK):
        super().__init__()
        in_f, out_f       = original.in_features, original.out_features
        self.in_features  = in_f; self.out_features = out_f
        self.original     = original
        self.lora_A       = nn.Linear(in_f, rank,  bias=False)
        self.lora_B       = nn.Linear(rank, out_f, bias=False)
        nn.init.kaiming_uniform_(self.lora_A.weight, a=0.01)
        nn.init.zeros_(self.lora_B.weight)
    def forward(self, x):
        return self.original(x) + self.lora_B(self.lora_A(x))


class FastLoRALinear(nn.Module):
    def __init__(self, slow, rank=LORA_RANK):
        super().__init__()
        in_f, out_f       = slow.in_features, slow.out_features
        self.in_features  = in_f; self.out_features = out_f
        self.slow         = slow
        self.fast_A       = nn.Linear(in_f, rank, bias=False)
        self.fast_B       = nn.Linear(rank, out_f, bias=False)
        nn.init.kaiming_uniform_(self.fast_A.weight, a=0.01)
        nn.init.zeros_(self.fast_B.weight)
    def forward(self, x):
        return self.slow(x) + self.fast_B(self.fast_A(x))


def inject_lora(encoder, block_indices, rank=LORA_RANK):
    for i, blk in enumerate(encoder.blocks):
        if i in block_indices:
            blk.attn.qkv = LoRALinear(blk.attn.qkv, rank=rank)
    print(f"  LoRA injected into blocks: {block_indices}")


def wrap_fast_lora(encoder, block_indices, rank=LORA_RANK):
    for i, blk in enumerate(encoder.blocks):
        if i in block_indices and isinstance(blk.attn.qkv, LoRALinear):
            blk.attn.qkv = FastLoRALinear(blk.attn.qkv, rank=rank)
    print(f"  Fast LoRA wrapped on blocks: {block_indices}")


def freeze_all(model):
    for p in model.parameters(): p.requires_grad = False


def unfreeze_lora_only(encoder):
    for name, p in encoder.named_parameters():
        p.requires_grad = ("lora_A" in name or "lora_B" in name)


def unfreeze_fast_lora(encoder):
    for name, p in encoder.named_parameters():
        p.requires_grad = ("fast_A" in name or "fast_B" in name)


def trainable_pct(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable, total, f"{100*trainable/total:.2f}%"


@torch.no_grad()
def ema_update(student, teacher, momentum=EMA_MOMENTUM):
    for ps, pt in zip(student.parameters(), teacher.parameters()):
        pt.data.mul_(momentum).add_((1 - momentum) * ps.data)


def save_ckpt(state, path):
    torch.save(state, path)
    print(f"  [ckpt] saved → {path}", flush=True)


def train_session(student, teacher, predictor, loader,
                   optimizer, scaler, session_idx, prefix):
    student.train(); predictor.train()
    epoch_losses = []
    for epoch in range(NUM_EPOCHS):
        total, n = 0.0, 0
        for videos in loader:
            videos = videos.to(DEVICE)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                loss = jepa_loss(student, teacher, predictor, videos, DEVICE)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(
                list(student.parameters()) + list(predictor.parameters()), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            ema_update(student, teacher)
            total += loss.item(); n += 1
        ep = total / max(n, 1)
        epoch_losses.append(ep)
        print(f"    epoch {epoch+1}/{NUM_EPOCHS} | loss {ep:.4f}", flush=True)
        save_ckpt(
            {"student": student.state_dict(), "teacher": teacher.state_dict(),
             "predictor": predictor.state_dict(), "session": session_idx, "epoch": epoch},
            os.path.join(CKPT_DIR, f"{prefix}_session{session_idx}_epoch{epoch}.pt")
        )
    return epoch_losses


print("Utilities ready.")

In [ ]:
# ── Cell 10: Experiment A — Vanilla FT ───────────────────────────────────────
print("=" * 60)
print("EXPERIMENT A: Vanilla Continual Fine-Tuning")
print("=" * 60)

student_A   = build_encoder().to(DEVICE)
teacher_A   = build_encoder().to(DEVICE)
teacher_A.load_state_dict(student_A.state_dict()); freeze_all(teacher_A)
predictor_A = JEPAPredictor(student_A.embed_dim).to(DEVICE)

optimizer_A = torch.optim.AdamW([
    {"params": student_A.parameters(),   "lr": LR_BACKBONE},
    {"params": predictor_A.parameters(), "lr": LR_PREDICTOR},
])
scaler_A  = torch.amp.GradScaler('cuda')
metrics_A = {}

for t, shard_range in enumerate(SESSION_SHARDS):
    print(f"\n  Session {t+1}/{len(SESSION_SHARDS)}", flush=True)
    loader  = make_loader(shard_range, max_clips=CLIPS_PER_SESSION)
    losses  = train_session(student_A, teacher_A, predictor_A,
                             loader, optimizer_A, scaler_A, t, "vanillaFT")
    res, embs_A, frames_A = evaluate(student_A)
    metrics_A[t] = {"losses": losses, **res}
    print(f"  Session {t+1} → Cycle@{EVAL_K}={res['cycle_at_k']:.3f}", flush=True)

bwt_A = metrics_A[2]["cycle_at_k"] - metrics_A[0]["cycle_at_k"]
print(f"\n[Exp A done] BWT={bwt_A:+.3f}")
print("=" * 60)

In [ ]:
# ── Cell 11: Experiment B — SSIAT/SAPT ───────────────────────────────────────
print("=" * 60)
print("EXPERIMENT B: SSIAT/SAPT — Shared Adapter")
print("=" * 60)

student_B = build_encoder()
inject_lora(student_B, TARGET_BLOCKS, rank=LORA_RANK)
student_B.to(DEVICE)
teacher_B = build_encoder()
inject_lora(teacher_B, TARGET_BLOCKS, rank=LORA_RANK)
teacher_B.to(DEVICE)
teacher_B.load_state_dict(student_B.state_dict()); freeze_all(teacher_B)
predictor_B = JEPAPredictor(student_B.embed_dim).to(DEVICE)

freeze_all(student_B)
unfreeze_lora_only(student_B)
predictor_B.requires_grad_(True)
tr, tot, pct = trainable_pct(student_B)
print(f"  Trainable: {tr:,} / {tot:,} ({pct})")

optimizer_B = torch.optim.AdamW([
    {"params": [p for p in student_B.parameters() if p.requires_grad], "lr": LR_BACKBONE},
    {"params": predictor_B.parameters(), "lr": LR_PREDICTOR},
])
scaler_B  = torch.amp.GradScaler('cuda')
metrics_B = {}

for t, shard_range in enumerate(SESSION_SHARDS):
    print(f"\n  Session {t+1}/{len(SESSION_SHARDS)}", flush=True)
    loader  = make_loader(shard_range, max_clips=CLIPS_PER_SESSION)
    losses  = train_session(student_B, teacher_B, predictor_B,
                             loader, optimizer_B, scaler_B, t, "ssiat")
    res, embs_B, frames_B = evaluate(student_B)
    metrics_B[t] = {"losses": losses, **res}
    print(f"  Session {t+1} → Cycle@{EVAL_K}={res['cycle_at_k']:.3f}", flush=True)

bwt_B = metrics_B[2]["cycle_at_k"] - metrics_B[0]["cycle_at_k"]
print(f"\n[Exp B done] BWT={bwt_B:+.3f}")
print("=" * 60)

In [ ]:
# ── Cell 12: Experiment C — SAFE ─────────────────────────────────────────────
print("=" * 60)
print("EXPERIMENT C: SAFE — Slow/Fast LoRA")
print("=" * 60)

student_C = build_encoder()
inject_lora(student_C, TARGET_BLOCKS, rank=LORA_RANK)
student_C.to(DEVICE)
teacher_C = build_encoder()
inject_lora(teacher_C, TARGET_BLOCKS, rank=LORA_RANK)
teacher_C.to(DEVICE)
teacher_C.load_state_dict(student_C.state_dict()); freeze_all(teacher_C)
predictor_C = JEPAPredictor(student_C.embed_dim).to(DEVICE)
metrics_C   = {}

# Stage 1: slow LoRA on session 1
print("\n── Stage 1: SLOW branch on session 1 ──")
freeze_all(student_C)
unfreeze_lora_only(student_C)
predictor_C.requires_grad_(True)
tr, tot, pct = trainable_pct(student_C)
print(f"  Slow trainable: {tr:,} / {tot:,} ({pct})")

opt_slow = torch.optim.AdamW([
    {"params": [p for p in student_C.parameters() if p.requires_grad], "lr": LR_BACKBONE},
    {"params": predictor_C.parameters(), "lr": LR_PREDICTOR},
])
scaler_C = torch.amp.GradScaler('cuda')

print(f"\n  Session 1/{len(SESSION_SHARDS)}", flush=True)
loader0  = make_loader(SESSION_SHARDS[0], max_clips=CLIPS_PER_SESSION)
losses0  = train_session(student_C, teacher_C, predictor_C,
                          loader0, opt_slow, scaler_C, 0, "safe")
freeze_all(student_C)
print("  [Slow branch frozen]", flush=True)
res, embs_C, frames_C = evaluate(student_C)
metrics_C[0] = {"losses": losses0, **res}
print(f"  Session 1 → Cycle@{EVAL_K}={res['cycle_at_k']:.3f}", flush=True)

# Stage 2: fast LoRA on sessions 2-3
print("\n── Stage 2: FAST branch on sessions 2-3 ──")
wrap_fast_lora(student_C, TARGET_BLOCKS, rank=LORA_RANK)
wrap_fast_lora(teacher_C, TARGET_BLOCKS, rank=LORA_RANK)
teacher_C.load_state_dict(student_C.state_dict()); freeze_all(teacher_C)
student_C.to(DEVICE); teacher_C.to(DEVICE)

unfreeze_fast_lora(student_C)
predictor_C.requires_grad_(True)
tr, tot, pct = trainable_pct(student_C)
print(f"  Fast trainable: {tr:,} / {tot:,} ({pct})")

opt_fast = torch.optim.AdamW([
    {"params": [p for p in student_C.parameters() if p.requires_grad], "lr": LR_BACKBONE},
    {"params": predictor_C.parameters(), "lr": LR_PREDICTOR},
])
scaler_C = torch.amp.GradScaler('cuda')

for t in range(1, len(SESSION_SHARDS)):
    print(f"\n  Session {t+1}/{len(SESSION_SHARDS)}", flush=True)
    loader  = make_loader(SESSION_SHARDS[t], max_clips=CLIPS_PER_SESSION)
    losses  = train_session(student_C, teacher_C, predictor_C,
                             loader, opt_fast, scaler_C, t, "safe")
    res, embs_C, frames_C = evaluate(student_C)
    metrics_C[t] = {"losses": losses, **res}
    print(f"  Session {t+1} → Cycle@{EVAL_K}={res['cycle_at_k']:.3f}", flush=True)

bwt_C = metrics_C[2]["cycle_at_k"] - metrics_C[0]["cycle_at_k"]
print(f"\n[Exp C done] BWT={bwt_C:+.3f}")
print("=" * 60)

In [ ]:
# ── Cell 13: Results table ────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("COMPARISON TABLE (3 methods, 2000 clips/session, JEPA loss)")
print("=" * 70)
print(f"{'Method':<15} {'S1 Cycle@10':<15} {'S2 Cycle@10':<15} {'S3 Cycle@10':<15} {'BWT':<10}")
print("-" * 70)

for name, metrics, bwt in [
    ("VanillaFT",  metrics_A, bwt_A),
    ("SSIAT/SAPT", metrics_B, bwt_B),
    ("SAFE",       metrics_C, bwt_C),
]:
    print(f"{name:<15} "
          f"{metrics[0]['cycle_at_k']:<15.3f} "
          f"{metrics[1]['cycle_at_k']:<15.3f} "
          f"{metrics[2]['cycle_at_k']:<15.3f} "
          f"{bwt:<10.3f}")

print("\nSEEKR reference (5000 clips/session):")
print(f"{'SEEKR':<15} {'0.809':<15} {'0.797':<15} {'0.793':<15} {'-0.016':<10}")
print("\nNote: SEEKR ran at 5000 clips/session; others at 2000.")

In [ ]:
# ── Cell 14: Combined t-SNE ───────────────────────────────────────────────────
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
titles    = ["Vanilla FT", "SSIAT/SAPT", "SAFE"]
emb_sets  = [embs_A, embs_B, embs_C]

for ax, title, embs in zip(axes, titles, emb_sets):
    print(f"Running t-SNE for {title}...", flush=True)
    coords = TSNE(n_components=2, random_state=42,
                  perplexity=min(30, len(embs)-1)).fit_transform(embs.numpy())
    sc = ax.scatter(coords[:, 0], coords[:, 1],
                    c=range(len(coords)), cmap='viridis', s=40, alpha=0.8)
    plt.colorbar(sc, ax=ax, label='Clip index')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('t-SNE dim 1'); ax.set_ylabel('t-SNE dim 2')

plt.suptitle('t-SNE of Encoder Embeddings — Final Session (walkindia val shard)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/workspace/tsne_combined.png', dpi=150)
plt.show()
print("t-SNE saved → /workspace/tsne_combined.png")

In [ ]:
# ── Cell 15: Combined neighbour grid ─────────────────────────────────────────
# One grid per method, all saved as one figure
fig, axes = plt.subplots(3 * N_QUERIES, N_NEIGHBOURS + 1,
                          figsize=(3 * (N_NEIGHBOURS + 1), 3 * 3 * N_QUERIES))

method_data = [
    ("Vanilla FT",  embs_A, frames_A),
    ("SSIAT/SAPT",  embs_B, frames_B),
    ("SAFE",        embs_C, frames_C),
]

for m_idx, (method_name, embs, frames) in enumerate(method_data):
    sim = embs @ embs.T
    sim.fill_diagonal_(-1e9)
    query_indices = np.linspace(0, len(embs)-1, N_QUERIES, dtype=int)

    for row_offset, q_idx in enumerate(query_indices):
        row = m_idx * N_QUERIES + row_offset
        axes[row, 0].imshow(frames[q_idx])
        label = f"{method_name}\nQ{row_offset+1}" if row_offset == 0 else f"Q{row_offset+1}"
        axes[row, 0].set_title(label, fontsize=8)
        axes[row, 0].axis('off')

        topk = sim[q_idx].topk(N_NEIGHBOURS).indices.tolist()
        for col, n_idx in enumerate(topk):
            axes[row, col + 1].imshow(frames[n_idx])
            axes[row, col + 1].set_title(f"NN{col+1}", fontsize=8)
            axes[row, col + 1].axis('off')

plt.suptitle('Query vs Top-5 Nearest Neighbours — All Methods (middle frame)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/workspace/neighbour_grid_combined.png', dpi=120)
plt.show()
print("Neighbour grid saved → /workspace/neighbour_grid_combined.png")

In [ ]:
# ── Cell 16: Package everything for download ──────────────────────────────────
import subprocess

# Zip checkpoints
result = subprocess.run(
    "tar -czf /workspace/all_checkpoints.tar.gz -C /workspace/checkpoints .",
    shell=True, capture_output=True, text=True
)
print("Checkpoints:", result.returncode)

# List output files
print("\nFiles ready for download:")
for f in ["/workspace/tsne_combined.png",
           "/workspace/neighbour_grid_combined.png",
           "/workspace/all_checkpoints.tar.gz"]:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1e6
        print(f"  {f}  ({size:.1f} MB)")

print("\nDownload via Jupyter file browser then destroy instance.")